In [0]:
dbutils.fs.ls("abfss://bronze@storagedatalake9105.dfs.core.windows.net/")


In [0]:
dbutils.fs.mkdirs("abfss://bronze@storagedatalake9105.dfs.core.windows.net/adventureworks/fact/sales")
dbutils.fs.ls("abfss://bronze@storagedatalake9105.dfs.core.windows.net/adventureworks/")


In [0]:
# Databricks - Bronze ingestion (raw CSV -> bronze Delta)
# Safe first run + incremental logic + schema registration on ADLS (not DBFS root).

from typing import Optional
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

ACCOUNT = "storagedatalake9105"

RAW_BASE = f"abfss://raw@{ACCOUNT}.dfs.core.windows.net/adventureworks"
BRONZE_BASE = f"abfss://bronze@{ACCOUNT}.dfs.core.windows.net/adventureworks"

RAW_SALES_GLOB = f"{RAW_BASE}/fact/sales_*.csv"
BRONZE_SALES_PATH = f"{BRONZE_BASE}/fact/sales"
BRONZE_DB_PATH = f"{BRONZE_BASE}/_metastore/bronze.db"
BRONZE_SALES_TABLE = "bronze.sales"

BUSINESS_KEYS = ["OrderNumber", "OrderLineItem"]
POTENTIAL_WATERMARK_COLUMNS = ["ModifiedDate", "OrderDate"]


def is_delta_table_path(path: str) -> bool:
    try:
        dbutils.fs.ls(f"{path}/_delta_log")
        return True
    except Exception:
        return False


def read_raw_sales() -> DataFrame:
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(RAW_SALES_GLOB)
    )


def with_ingestion_columns(df: DataFrame) -> DataFrame:
    return (
        df.withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.input_file_name())
          .withColumn("_batch_id", F.date_format(F.current_timestamp(), "yyyyMMddHHmmss"))
    )


def choose_watermark_column(df: DataFrame) -> Optional[str]:
    columns = set(df.columns)
    for col_name in POTENTIAL_WATERMARK_COLUMNS:
        if col_name in columns:
            return col_name
    return None


def incremental_filter(df: DataFrame, bronze_path: str) -> DataFrame:
    # First load (or non-delta path): load all
    if not is_delta_table_path(bronze_path):
        return df

    current_df = spark.read.format("delta").load(bronze_path)
    watermark_column = choose_watermark_column(df)

    # Watermark-based incremental
    if watermark_column and watermark_column in current_df.columns:
        max_ts = current_df.select(F.max(F.col(watermark_column)).alias("max_ts")).first()["max_ts"]
        if max_ts is not None:
            return df.filter(F.col(watermark_column) > F.lit(max_ts))
        return df

    # Business-key incremental fallback
    if all(k in df.columns for k in BUSINESS_KEYS) and all(k in current_df.columns for k in BUSINESS_KEYS):
        existing_keys = current_df.select(*BUSINESS_KEYS).dropDuplicates()
        return df.join(existing_keys, BUSINESS_KEYS, "left_anti")

    return df


def register_bronze_objects() -> None:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS bronze LOCATION '{BRONZE_DB_PATH}'")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {BRONZE_SALES_TABLE}
        USING DELTA
        LOCATION '{BRONZE_SALES_PATH}'
    """)


def main() -> None:
    print(f"Reading raw sales from: {RAW_SALES_GLOB}")
    raw_df = read_raw_sales()
    print(f"Raw rows read: {raw_df.count()}")

    enriched_df = with_ingestion_columns(raw_df)
    new_rows_df = incremental_filter(enriched_df, BRONZE_SALES_PATH)
    load_count = new_rows_df.count()

    print(f"Rows to load into bronze: {load_count}")

    if load_count > 0:
        (
            new_rows_df.write
            .format("delta")
            .mode("append")
            .save(BRONZE_SALES_PATH)
        )

    register_bronze_objects()

    print(f"Bronze ingest complete. Rows loaded: {load_count}")
    print(f"Bronze table: {BRONZE_SALES_TABLE}")
    print(f"Bronze path: {BRONZE_SALES_PATH}")


if __name__ == "__main__":
    main()


In [0]:
%sql
SELECT COUNT(*) FROM bronze.sales;


In [0]:
# Databricks - Bronze dimension ingestion (raw CSV -> bronze Delta)

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

ACCOUNT = "storagedatalake9105"

RAW_BASE = f"abfss://raw@{ACCOUNT}.dfs.core.windows.net/adventureworks"
BRONZE_BASE = f"abfss://bronze@{ACCOUNT}.dfs.core.windows.net/adventureworks"
BRONZE_DB_PATH = f"{BRONZE_BASE}/_metastore/bronze.db"

RAW_DIM_BASE = f"{RAW_BASE}/dim"
BRONZE_DIM_BASE = f"{BRONZE_BASE}/dim"

# raw_file_name -> bronze_table_name
DIM_FILES = {
    "calendar.csv": "dim_calendar",
    "customers.csv": "dim_customers",
    "product_categories.csv": "dim_product_categories",
    "product_subcategories.csv": "dim_product_subcategories",
    "products.csv": "dim_products",
    "territories.csv": "dim_territories",
}


def read_csv(path: str) -> DataFrame:
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(path)
    )


def add_ingestion_cols(df: DataFrame) -> DataFrame:
    return (
        df.withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.input_file_name())
          .withColumn("_batch_id", F.date_format(F.current_timestamp(), "yyyyMMddHHmmss"))
    )


def ingest_one(raw_file: str, table_name: str) -> None:
    raw_path = f"{RAW_DIM_BASE}/{raw_file}"
    bronze_path = f"{BRONZE_DIM_BASE}/{table_name}"

    df = read_csv(raw_path)
    df = add_ingestion_cols(df)

    # Bronze as latest raw snapshot per file
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(bronze_path)
    )

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS bronze.{table_name}
        USING DELTA
        LOCATION '{bronze_path}'
    """)

    print(f"Loaded bronze.{table_name}: {df.count()} rows")


def main() -> None:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS bronze LOCATION '{BRONZE_DB_PATH}'")

    for raw_file, table_name in DIM_FILES.items():
        ingest_one(raw_file, table_name)

    print("Bronze dimension ingestion complete.")


if __name__ == "__main__":
    main()
